In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import numpy as np
import math
import seaborn as sns
import pyarrow as pa
import pyarrow.parquet as pq
import itertools
import evi_functions as evi_func
import matplotlib.dates as mdates
import warnings
from pathlib import Path
from datetime import datetime

import descri_function as des_fun

In [2]:
df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/sintetic_outbreak_20260325.parquet')

df_meta = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/sintetic_outbreak_metadata_20260325.parquet')

In [3]:
df.columns

Index(['replicate_0', 'replicate_1', 'replicate_2', 'replicate_3',
       'replicate_4', 'replicate_5', 'replicate_6', 'replicate_7',
       'replicate_8', 'replicate_9', 'replicate_10', 'replicate_11',
       'replicate_12', 'replicate_13', 'replicate_14', 'replicate_15',
       'replicate_16', 'replicate_17', 'replicate_18', 'replicate_19',
       'replicate_20', 'replicate_21', 'replicate_22', 'replicate_23',
       'replicate_24', 'replicate_25', 'replicate_26', 'replicate_27',
       'replicate_28', 'replicate_29', 'replicate_30', 'replicate_31',
       'co_ibge', 'year_week', 'atend_ivas',
       'mem_surge_01_correct_with_consec', 'warning_final_mem_surge_01'],
      dtype='object')

In [4]:
df.year_week.min()

'2017-01'

In [5]:
df.year_week.max()

'2025-32'

In [6]:
dta = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/cities_valid_for_MEM_26_03_2026.parquet')

# Select cities for the manuscript analysis (valid MEM)
#lst = list(set(df.co_ibge.unique()) - set(dta.co_ibge.unique()))
lst = dta.co_ibge.unique()
df = df[df.co_ibge.isin(lst)]

df =  df[(df.year_week >= '2022-42') &(df.year_week <= '2025-32')]

In [7]:
# Select cities for the manuscript analysis (valid MEM)
df_meta = df_meta[df_meta.co_ibge.isin(lst)]


In [8]:
df_meta.start_year_week.min()

'2024-51'

In [9]:
df_meta = df_meta.assign(ones = 1)

In [10]:
#sint_surges_count = df_meta.groupby('replicate')['ones'].sum().reset_index()

In [11]:
df_meta.columns

Index(['start', 'end', 'size', 'k', 'duration', 'replicate', 'co_ibge',
       'start_year_week', 'end_year_week', 'ones'],
      dtype='object')

In [12]:
df_meta.groupby(['co_ibge','replicate'])['ones'].sum().reset_index().ones.max()

2

In [13]:
df_meta.groupby(['co_ibge','replicate'])['ones'].sum().reset_index().ones.min()

1

In [20]:
#df_meta.groupby(['replicate'])['ones'].sum().reset_index()#.ones.sum()

# Criar coluna de surtos artificiais para cada replica de cada municipio

In [15]:
# convert to weekly period (ISO-like)
df = df.assign(yw = pd.to_datetime(df['year_week'] + '-1', format='%Y-%W-%w').dt.to_period('W'))
df_meta = df_meta.assign(start_yw = pd.to_datetime(df_meta['start_year_week'] + '-1', format='%Y-%W-%w').dt.to_period('W'))
df_meta = df_meta.assign(end_yw   = pd.to_datetime(df_meta['end_year_week'] + '-1', format='%Y-%W-%w').dt.to_period('W'))

In [16]:
# esta função é para identificar os surtos inseridos independentemente se coincidem ou não com um surto do MEM
#def compute_replicates(df, df_meta):
#   df = df.copy()

#    for code, set_muni in df.groupby('co_ibge'):
#        meta_city = df_meta[df_meta['co_ibge'] == code]#
#
#       for rep, dta in meta_city.groupby('replicate'):#
#
#            cond_consec = (
#                (set_muni['yw'].values[:, None] >= dta['start_yw'].values) &
#                (set_muni['yw'].values[:, None] <= dta['end_yw'].values)
#           ).any(axis=1)
#
#            cond_start = set_muni['yw'].isin(dta['start_yw'])
#
#            df.loc[set_muni.index, f'replicate_{rep}_surge_consec'] = cond_consec.astype(int)
#            df.loc[set_muni.index, f'replicate_{rep}_surge'] = cond_start.astype(int)
#
#    return df

In [17]:
def compute_replicates(df, df_meta):
    df = df.copy()

    for code, set_muni in df.groupby('co_ibge'):
        meta_city = df_meta[df_meta['co_ibge'] == code]

        warning = set_muni['warning_final_mem_surge_01'].values
        yw = set_muni['yw'].values

        for rep, dta in meta_city.groupby('replicate'):

            # --- STEP 1: identify NON-coincident starts ---
            is_start = set_muni['yw'].isin(dta['start_yw']).values
            noncoincident_start = is_start & (warning == 0)

            # store start signal
            df.loc[set_muni.index, f'replicate_{rep}_surge'] = noncoincident_start.astype(int)

            # --- STEP 2: keep only intervals whose start is non-coincident ---
            valid_intervals = dta[dta['start_yw'].isin(set_muni.loc[noncoincident_start, 'yw'])]

            if len(valid_intervals) == 0:
                df.loc[set_muni.index, f'replicate_{rep}_surge_consec'] = 0
                continue

            # build consecutive mask WITHOUT warning condition
            cond_consec = (
                (yw[:, None] >= valid_intervals['start_yw'].values) &
                (yw[:, None] <= valid_intervals['end_yw'].values)
            ).any(axis=1)

            df.loc[set_muni.index, f'replicate_{rep}_surge_consec'] = cond_consec.astype(int)

    return df

In [30]:
result = compute_replicates(df, df_meta)

In [ ]:
#result[result.co_ibge == 522230][['co_ibge', 'year_week', 
#       'mem_surge_01_correct_with_consec', 
#       'warning_final_mem_surge_01', 
#       'replicate_3_surge_consec','replicate_10_surge']][-35:]

In [ ]:
#result.to_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/sintetic_with_surge_col_28_04_2026_noncoincident_start.parquet')

In [33]:
result = result[result.year_week >= '2024-51']

In [34]:
rep_cols = [c for c in result.columns if c.startswith('replicate_') and c.endswith('_surge')]

counts = {
    col: ((result[col] == 1)).sum()
    for col in rep_cols
}

counts_df = pd.DataFrame.from_dict(counts, orient='index', columns=['non_coincident_surges'])

In [35]:
print('Mean (per replicate) of total sintetic onset surges with 95% CI that does not coincide with onset surges in MEM', des_fun.mean_confidence_interval(
   counts_df['non_coincident_surges'], confidence=0.95
))

Mean (per replicate) of total sintetic surges with 95% CI that does not coincide with onset surges in MEM (7627, 7396, 7858)


In [108]:
rep_cols2 = [c for c in result.columns if c.startswith('replicate_') and c.endswith('_surge_consec')]

counts2 = {
    col: ((result[col] == 1)).sum()
    for col in rep_cols2
}

counts_df2 = pd.DataFrame.from_dict(counts2, orient='index', columns=['non_coincident_surges'])

In [109]:
print('Mean (per replicate) of total sintetic surges with 95% CI that does not coincide with onset surges in MEM', des_fun.mean_confidence_interval(
   counts_df2['non_coincident_surges'], confidence=0.95
))

Mean (per replicate) of total sintetic surges with 95% CI that does not coincide with onset surges in MEM (59806, 58269, 61342)


# Classificar intensidade dos surtos sinteticos

In [36]:
# Classify MEM surges accordingly to intensity
df_mem = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/mem_output_26_03_2026.parquet')

df_mem = df_mem[df_mem.epiyear.isin([2022,2023, 2024,2025])]

dta_intens = df_mem.groupby(['co_ibge'])[['baseline', 'post_baseline', 'epidemic_threshold',
       'post_threshold', 'low_level', 'medium_level', 'high_level']].max().reset_index()

In [37]:
# =============================================================================
# FUNCTION TO ASSIGN INTENSITY
# =============================================================================

def classify_block(values, base, epi, low, med, high):

    # Check from most severe to least

    if (values > high).any():
        return 'very high'

    elif (values > med).any():
        return 'high'

    elif (values > low).any():
        return 'medium'

    elif (values > epi).any():
        return 'low'

    elif ((values > base) & (values <= epi)).any():
        return 'very low'

    elif (values <= base).all():
        return 'baseline'

    return np.nan


# =============================================================================
# CREATE INTENSITY LABELS
# =============================================================================

def ints_creator(df, dta_intens):

    lst = []

    # Automatically detect replicate columns
    rep_cols = [
        c for c in df.columns
        if c.startswith('replicate_') and c.endswith('_surge_consec')
    ]

    for code, set_muni in df.groupby('co_ibge'):

        print(f'Processing municipality: {code}')

        set_muni = (
            set_muni
            .copy()
            .sort_values('year_week')
        )

        # Municipality-specific thresholds
        set_muni_inten = dta_intens[
            dta_intens.co_ibge == code
        ]

        # Skip if thresholds do not exist
        if set_muni_inten.empty:

            #print(f'No thresholds found for {code}')
            lst.append(set_muni)
            continue

        thresholds = set_muni_inten.iloc[0]

        # Thresholds
        base = int(thresholds['baseline'])
        epi  = int(thresholds['epidemic_threshold'])

        low = (
            int(thresholds['low_level'])
            if pd.notna(thresholds.get('low_level'))
            else 2 * epi
        )

        med = (
            int(thresholds['medium_level'])
            if pd.notna(thresholds.get('medium_level'))
            else 4 * epi
        )

        high = (
            int(thresholds['high_level'])
            if pd.notna(thresholds.get('high_level'))
            else 6 * epi
        )

        print(
            f'base={base}, epi={epi}, low={low}, med={med}, high={high}'
        )

        # ---------------------------------------------------------------------
        # Process each replicate column
        # ---------------------------------------------------------------------

        for col in rep_cols:

            #print(f'Processing {col}')

            # Extract replicate name
            rep_id = col.replace('_surge_consec', '')

            # Binary signal
            flag = set_muni[col]

            # Create block IDs for consecutive sequences
            block = (flag != flag.shift()).cumsum()

            intensity_col = f'intensity_{rep_id}'

            set_muni[intensity_col] = np.nan

            # -----------------------------------------------------------------
            # Process only positive blocks
            # -----------------------------------------------------------------

            for block_id, group in set_muni.groupby(block):

                # Skip non-surge blocks
                if group[col].iloc[0] != 1:
                    continue

                values = group['atend_ivas']

                # Assign intensity
                intensity_label = classify_block(
                    values,
                    base,
                    epi,
                    low,
                    med,
                    high
                )

                set_muni.loc[
                    group.index,
                    intensity_col
                ] = intensity_label

        lst.append(set_muni)

    return pd.concat(lst, ignore_index=True)

In [38]:
import warnings
warnings.filterwarnings('ignore')

res_int = ints_creator(result, dta_intens)

Processing municipality: 110001
base=28, epi=49, low=55, med=92, high=111
Processing municipality: 110002
base=53, epi=96, low=105, med=145, high=164
Processing municipality: 110005
base=20, epi=45, low=53, med=112, high=145
Processing municipality: 110007
base=5, epi=15, low=19, med=32, high=40
Processing municipality: 110008
base=10, epi=22, low=29, med=39, high=45
Processing municipality: 110009
base=41, epi=61, low=82, med=138, high=174
Processing municipality: 110010
base=45, epi=68, low=102, med=126, high=150
Processing municipality: 110011
base=68, epi=112, low=133, med=167, high=188
Processing municipality: 110013
base=45, epi=105, low=106, med=162, high=184
Processing municipality: 110014
base=32, epi=45, low=75, med=81, high=86
Processing municipality: 110015
base=66, epi=121, low=143, med=154, high=166
Processing municipality: 110018
base=119, epi=204, low=210, med=316, high=354
Processing municipality: 110020
base=364, epi=641, low=646, med=1301, high=1591
Processing munici

In [39]:
# detect replicate IDs from intensity columns
rep_ids = [
    col.replace('intensity_replicate_', '')
    for col in res_int.columns
    if col.startswith('intensity_replicate_')
]

for rep in rep_ids:

    intensity_col = f'intensity_replicate_{rep}'
    surge_col = f'replicate_{rep}_surge'
    consec_col = f'replicate_{rep}_surge_consec'

    # --- NON-consecutive ---
    res_int[f'mem_low_rep_{rep}'] = (
        (res_int[intensity_col] == 'low') &
        (res_int[surge_col] == 1)
    ).astype(int)

    res_int[f'mem_medium_rep_{rep}'] = (
        (res_int[intensity_col] == 'medium') &
        (res_int[surge_col] == 1)
    ).astype(int)

    res_int[f'mem_high_rep_{rep}'] = (
        (res_int[intensity_col] == 'high') &
        (res_int[surge_col] == 1)
    ).astype(int)

    res_int[f'mem_very_high_rep_{rep}'] = (
        (res_int[intensity_col] == 'very high')
        & (res_int[surge_col] == 1)
    ).astype(int)

    # --- CONSECUTIVE ---
    res_int[f'mem_low_consec_rep_{rep}'] = (
        (res_int[intensity_col] == 'low') &
        (res_int[consec_col] == 1)
    ).astype(int)

    res_int[f'mem_medium_consec_rep_{rep}'] = (
        (res_int[intensity_col] == 'medium') &
        (res_int[consec_col] == 1)
    ).astype(int)

    res_int[f'mem_high_consec_rep_{rep}'] = (
        (res_int[intensity_col] == 'high') &
        (res_int[consec_col] == 1)
    ).astype(int)

    res_int[f'mem_very_high_consec_rep_{rep}'] = (
        (res_int[intensity_col] == 'very high') &
        (res_int[consec_col] == 1)
    ).astype(int)

In [40]:
len(res_int.columns.to_list())

390

# Count the sintetic surges by intensity

In [110]:
levels = [
    'baseline',
    'very low',
    'low',
    'medium',
    'high',
    'very high'
]

lst = []

for rep in range(32):

    dta = (
        res_int
        .groupby(f'intensity_replicate_{rep}')[f'replicate_{rep}_surge']
        .sum()
        .reindex(levels, fill_value=0)
        .to_frame()
        .T
    )

    dta.index = [f'replicate_{rep}']

    lst.append(dta)

final_intensity = pd.concat(lst)

In [113]:
final_intensity = final_intensity.assign(under_mem = final_intensity.baseline + final_intensity['very low'])

In [120]:
for cols in ['low', 'medium', 'high', 'very high','under_mem']:

    print('Mean surges', cols, des_fun.mean_confidence_interval(
       final_intensity[cols], confidence=0.95
        ))

Mean surges low (1240, 1213, 1267)
Mean surges medium (862, 840, 883)
Mean surges high (371, 359, 383)
Mean surges very high (1006, 960, 1052)
Mean surges under_mem (4149, 3923, 4376)
